In [1]:
from deepagents.backends import LocalShellBackend

root_dir= "/home/cooper/githubProjects/agent-deploy-kit/agents/data_agent_langchain"
backend = LocalShellBackend(root_dir=root_dir, env={"PATH": "/usr/bin:/bin"})

/tmp/ipykernel_11388/752993861.py:4: LangChainDeprecationWarning: `LocalShellBackend` `virtual_mode` default will change in deepagents==0.6.0; please specify `virtual_mode` explicitly. Note: `virtual_mode` is for virtual path semantics (e.g., `CompositeBackend` routing) and optional path-based guardrails; it does not provide sandboxing or process isolation. Security note: leaving `virtual_mode=False` allows absolute paths and `'..'` to bypass `root_dir`, and `LocalShellBackend` provides no sandboxing (`execute()` runs commands on the host; `virtual_mode` does not restrict shell execution). See https://reference.langchain.com/python/deepagents/ for usage guidelines.
  backend = LocalShellBackend(root_dir=root_dir, env={"PATH": "/usr/bin:/bin"})


In [2]:
import csv
import io

# Create sample sales data
data = [
    ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
]

# Convert to CSV bytes
text_buf = io.StringIO()
writer = csv.writer(text_buf)
writer.writerows(data)
csv_bytes = text_buf.getvalue().encode("utf-8")
text_buf.close()

# Upload to backend
backend.upload_files([(f"{root_dir}/data/sales_data.csv", csv_bytes)])

[FileUploadResponse(path='/home/cooper/githubProjects/agent-deploy-kit/agents/data_agent_langchain/data/sales_data.csv', error=None)]

In [ ]:
from utils.langchain_model import get_singleton_client
from langchain.agents import create_agent
from deepagents.middleware import FilesystemMiddleware, SummarizationMiddleware, SkillsMiddleware, SubAgentMiddleware
from langchain.agents.middleware import TodoListMiddleware
from deepagents import SubAgent

model=get_singleton_client(llm_provider="longcat")

visualizer: SubAgent = {
    "name": "visualizer",
    "description": "Generates charts and visualizations from data files in the sandbox.",
    "system_prompt": "You are a data visualization specialist. Write Python scripts using matplotlib and seaborn. Save all figures as PNG files.",
    "tools": [],
    "model": model,
}

agent = create_agent(
    model=model,
    tools=[],
    middleware=[FilesystemMiddleware(backend=backend),
                SummarizationMiddleware(model=model, backend=backend),
                SkillsMiddleware(backend=backend, sources=["/skills/"],),
                TodoListMiddleware(),
                SubAgentMiddleware(backend=backend, subagents=[visualizer]),
                ],
)



input_message = {
    "role": "user",
    "content": (
       "Read /sales.csv and summarize total revenue by product in one sentence. Do not run shell commands."
    ),
}


for chunk in agent.stream(
        {'messages': [input_message]},
        config={"recursion_limit": 8},
        ):
    print(chunk)


{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 2751, 'total_tokens': 2769, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'image_tokens': 0, 'video_tokens': 0, 'text_tokens': 0}, 'effectiveCachedTokens': 0, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'output_tokens_details': None, 'cached_tokens': 0}, 'model_provider': 'openai', 'model_name': 'LongCat-2.0-Preview', 'system_fingerprint': None, 'id': '2a63a1a02e8846cfa8b0b2d77db3e183', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ee52a-1fdc-78f3-a35d-b25a3c2393e8-0', tool_calls=[{'name': 'read_file', 'args': {'file_path': '/sales.csv'}, 'id': 'call_60dad40205f04db98aa5664f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2751, 'output_tokens': 18, 'total_tokens': 2769, 'input_token